In [8]:
# Линейная регрессия (OLS) и взаимодействия признаков
# Цель: Построить модель цены дома, добавить взаимодействия с категориальными переменными

import pandas as pd
import numpy as np
import statsmodels.api as sm
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

try:
    df = pd.read_csv('kc_house_data.csv')
    print(f"Данные загружены. Размер: {df.shape}")
except FileNotFoundError:
    print("Ошибка: файл 'kc_house_data.csv' не найден в текущей папке.")
    print("Текущая директория:", os.getcwd())
    raise

Данные загружены. Размер: (21613, 21)


In [11]:
print('ПОСТРОЕНИЕ ЛИНЕЙНОЙ МОДЕЛИ ДЛЯ ЦЕНЫ (price)')
numeric_features = ['bedrooms', 'bathrooms', 'sqft_living', 'sqft_lot', 'floors',
                     'waterfront', 'condition', 'grade', 'sqft_above',
                     'sqft_basement', 'yr_built', 'yr_renovated', 'lat', 'long',
                     'sqft_living15', 'sqft_lot15']

X = df[numeric_features].astype(float)
y = df['price'].astype(float)

X_with_const = sm.add_constant(X)

model_1 = sm.OLS(y, X_with_const).fit()

print("БАЗОВАЯ ЛИНЕЙНАЯ МОДЕЛЬ")
print(f"R²: {model_1.rsquared:.4f}")
print(f"Adjusted R²: {model_1.rsquared_adj:.4f}")

coef_df = pd.DataFrame({
    'feature': model_1.params.index,
    'coef': model_1.params.values,
    'p_value': model_1.pvalues.values
})
significant = coef_df[coef_df['p_value'] < 0.05].iloc[1:]  # пропускаем const

print("\nЗначимые признаки (p-value < 0.05):")
print(significant[['feature', 'coef']].to_string(index=False))

ПОСТРОЕНИЕ ЛИНЕЙНОЙ МОДЕЛИ ДЛЯ ЦЕНЫ (price)
БАЗОВАЯ ЛИНЕЙНАЯ МОДЕЛЬ
R²: 0.6879
Adjusted R²: 0.6877

Значимые признаки (p-value < 0.05):
      feature           coef
     bedrooms  -37611.016422
    bathrooms   43478.574049
  sqft_living     113.645360
     sqft_lot       0.165466
   waterfront  738774.583101
    condition   31351.854442
        grade  101253.716465
   sqft_above      64.803286
sqft_basement      48.843784
     yr_built   -2613.350259
 yr_renovated      23.991737
          lat  543311.363570
         long -142273.951173
sqft_living15      39.384136
   sqft_lot15      -0.384705


In [12]:
print('ДОБАВЛЕНИЕ ВЗАИМОДЕЙСТВИЙ С КАТЕГОРИАЛЬНОЙ ПЕРЕМЕННОЙ view')
print("Уникальные значения view:", sorted(df['view'].unique()))
print("\nРаспределение view:")
print(df['view'].value_counts().sort_index())

view_dummies = pd.get_dummies(df['view'], prefix='view', drop_first=True).astype(float)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled = pd.DataFrame(X_scaled, columns=numeric_features, index=X.index)

coef_abs = np.abs(model_1.params.iloc[1:])
top_features = coef_abs.nlargest(3).index.tolist()
print(f"\nПризнаки для взаимодействий (топ-3 по |коэффициенту|): {top_features}")

X_with_interactions = X_scaled.copy()

for col in view_dummies.columns:
    X_with_interactions[col] = view_dummies[col]

for feature in top_features:
    for view_val in view_dummies.columns:
        interaction_name = f'{feature}_x_{view_val}'
        X_with_interactions[interaction_name] = X_scaled[feature] * view_dummies[view_val]

X_with_interactions = X_with_interactions.astype(float)

X_with_interactions_const = sm.add_constant(X_with_interactions)
model_2 = sm.OLS(y, X_with_interactions_const).fit()

print(f"\nR² модели с взаимодействиями: {model_2.rsquared:.4f}")
print(f"Улучшение R²: +{(model_2.rsquared - model_1.rsquared)*100:.2f}%")

ДОБАВЛЕНИЕ ВЗАИМОДЕЙСТВИЙ С КАТЕГОРИАЛЬНОЙ ПЕРЕМЕННОЙ view
Уникальные значения view: [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4)]

Распределение view:
view
0    19489
1      332
2      963
3      510
4      319
Name: count, dtype: int64

Признаки для взаимодействий (топ-3 по |коэффициенту|): ['waterfront', 'lat', 'long']

R² модели с взаимодействиями: 0.7051
Улучшение R²: +1.72%


In [14]:
print("СРАВНЕНИЕ НА ОБУЧАЮЩЕЙ И ТЕСТОВОЙ ВЫБОРКАХ")

X_train, X_test, y_train, y_test = train_test_split(
    X_with_interactions_const, y, test_size=0.3, random_state=1
)

X_scaled_const = sm.add_constant(X_scaled)
X_train_scaled, X_test_scaled, _, _ = train_test_split(
    X_scaled_const, y, test_size=0.3, random_state=1
)

model_1_train = sm.OLS(y_train, X_train_scaled).fit()
model_2_train = sm.OLS(y_train, X_train).fit()

y_train_pred_1 = model_1_train.predict(X_train_scaled)
y_test_pred_1 = model_1_train.predict(X_test_scaled)

y_train_pred_2 = model_2_train.predict(X_train)
y_test_pred_2 = model_2_train.predict(X_test)

results = pd.DataFrame({
    'Модель': ['Без взаимодействий', 'С взаимодействиями'],
    'R² на обучении': [
        r2_score(y_train, y_train_pred_1),
        r2_score(y_train, y_train_pred_2)
    ],
    'R² на тесте': [
        r2_score(y_test, y_test_pred_1),
        r2_score(y_test, y_test_pred_2)
    ]
})

results['Разница (тест - обучение)'] = results['R² на тесте'] - results['R² на обучении']

print(results.to_string(index=False))

diff_1 = results.loc[0, 'Разница (тест - обучение)']
diff_2 = results.loc[1, 'Разница (тест - обучение)']

if diff_1 < 0:
    print(f"Модель 1 (без взаимодействий): тест хуже обучения на {abs(diff_1)*100:.2f}% → есть переобучение")
else:
    print(f"Модель 1 (без взаимодействий): тест лучше обучения")

if diff_2 < 0:
    print(f"Модель 2 (с взаимодействиями): тест хуже обучения на {abs(diff_2)*100:.2f}% → есть переобучение")
else:
    print(f"Модель 2 (с взаимодействиями): тест лучше обучения")

СРАВНЕНИЕ НА ОБУЧАЮЩЕЙ И ТЕСТОВОЙ ВЫБОРКАХ
            Модель  R² на обучении  R² на тесте  Разница (тест - обучение)
Без взаимодействий        0.692080     0.678626                  -0.013454
С взаимодействиями        0.711159     0.689553                  -0.021606
Модель 1 (без взаимодействий): тест хуже обучения на 1.35% → есть переобучение
Модель 2 (с взаимодействиями): тест хуже обучения на 2.16% → есть переобучение


In [15]:
print("ВЫВОДЫ")

best_model = "С взаимодействиями" if model_2.rsquared > model_1.rsquared else "Без взаимодействий"
print(f"1. Лучшая модель по R²: {best_model}")
print(f"2. Добавление взаимодействий улучшило R² на {(model_2.rsquared - model_1.rsquared)*100:.2f}%")

if diff_2 < diff_1:
    print("3. Модель с взаимодействиями показала меньшее переобучение")
else:
    print("3. Базовая модель показала меньшее переобучение")

print("\nКлючевые выводы по значимым признакам:")
for _, row in significant.head(5).iterrows():
    print(f"  • {row['feature']}: {row['coef']:,.0f} руб. (p={row['p_value']:.2e})")

ВЫВОДЫ
1. Лучшая модель по R²: С взаимодействиями
2. Добавление взаимодействий улучшило R² на 1.72%
3. Модель с взаимодействиями показала меньшее переобучение

Ключевые выводы по значимым признакам:
  • bedrooms: -37,611 руб. (p=1.12e-84)
  • bathrooms: 43,479 руб. (p=3.96e-39)
  • sqft_living: 114 руб. (p=0.00e+00)
  • sqft_lot: 0 руб. (p=7.03e-04)
  • waterfront: 738,775 руб. (p=0.00e+00)
